In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("data/customer_churn.csv")

print("Customer Churn Analysis")
print("=" * 50)

print("\nDataset shape:")
print(df.shape)

print("\nFirst five rows:")
display(df.head())

print("Dataset columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nSummary statistics:")
display(df.describe())

# Convert TotalCharges to numeric if the column exists
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"],
        errors="coerce"
    )

# Fill missing numeric values with median
numeric_columns = df.select_dtypes(include=["number"]).columns

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Fill missing categorical values with mode
categorical_columns = df.select_dtypes(include=["object"]).columns

for column in categorical_columns:
    if df[column].isnull().sum() > 0:
        df[column] = df[column].fillna(df[column].mode()[0])

print("Missing values after cleaning:")
print(df.isnull().sum())

print("\nCleaned dataset shape:")
print(df.shape)

if "Churn" in df.columns:
    print("Churn counts:")
    print(df["Churn"].value_counts())

    print("\nChurn percentages:")
    churn_percent = df["Churn"].value_counts(normalize=True) * 100
    print(churn_percent.round(2))

plt.figure(figsize=(7, 5))

df["Churn"].value_counts().plot(kind="bar")

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.tight_layout()

plt.show()

if "MonthlyCharges" in df.columns:

    plt.figure(figsize=(8, 5))

    for churn_value in df["Churn"].unique():
        subset = df[df["Churn"] == churn_value]

        plt.hist(
            subset["MonthlyCharges"],
            bins=30,
            alpha=0.5,
            label=f"Churn = {churn_value}"
        )

    plt.title("Monthly Charges by Churn Status")
    plt.xlabel("Monthly Charges")
    plt.ylabel("Number of Customers")
    plt.legend()
    plt.tight_layout()

    plt.show()

if "tenure" in df.columns:

    churn_tenure = df.groupby("Churn")["tenure"].mean()

    print("Average customer tenure by churn status:")
    print(churn_tenure.round(2))

    churn_tenure.plot(kind="bar")

    plt.title("Average Customer Tenure by Churn Status")
    plt.xlabel("Churn")
    plt.ylabel("Average Tenure")
    plt.xticks(rotation=0)
    plt.tight_layout()

    plt.show()

model_df = df.copy()

# Remove customer identifier because it does not help predict churn
if "customerID" in model_df.columns:
    model_df = model_df.drop(columns=["customerID"])

encoder = LabelEncoder()

# Encode categorical variables
for column in model_df.select_dtypes(include=["object"]).columns:
    model_df[column] = encoder.fit_transform(
        model_df[column].astype(str)
    )

print("Prepared modeling dataset:")
display(model_df.head())

print("\nModeling dataset shape:")
print(model_df.shape)

X = model_df.drop(columns=["Churn"])
y = model_df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print(f"Model Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions
    )
)

cm = confusion_matrix(
    y_test,
    predictions
)

confusion_df = pd.DataFrame(
    cm,
    columns=["Predicted No", "Predicted Yes"],
    index=["Actual No", "Actual Yes"]
)

print("Confusion Matrix:")
display(confusion_df)

feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(
    ascending=False
)

print("Top 10 predictive features:")
display(feature_importance.head(10))

top_features = feature_importance.head(10)

plt.figure(figsize=(9, 6))

top_features.sort_values().plot(kind="barh")

plt.title("Top 10 Features Affecting Customer Churn")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.tight_layout()

plt.show()

print("KEY FINDINGS")
print("=" * 50)

churn_rate = (
    df["Churn"]
    .value_counts(normalize=True)
    .get("Yes", 0)
    * 100
)

print(f"1. Overall churn rate: {churn_rate:.2f}%")

if "tenure" in df.columns:
    tenure_summary = df.groupby("Churn")["tenure"].mean()

    if "Yes" in tenure_summary.index:
        print(
            f"2. Customers who churned had an average tenure of "
            f"{tenure_summary['Yes']:.2f} months."
        )

    if "No" in tenure_summary.index:
        print(
            f"3. Customers who stayed had an average tenure of "
            f"{tenure_summary['No']:.2f} months."
        )

print(f"4. Random Forest model accuracy: {accuracy:.2%}")

print("\nTop predictive features:")

for feature, importance in feature_importance.head(5).items():
    print(f"- {feature}: {importance:.4f}")

print("""
PROJECT SUMMARY

This project used a Kaggle customer churn dataset to perform
data cleaning, exploratory data analysis, visualization, and
machine learning.

A Random Forest classifier was trained to predict whether a
customer would churn based on customer account, service, and
billing characteristics.

The project demonstrates experience with:

- Python
- Pandas
- Data Cleaning
- Exploratory Data Analysis
- Statistical Analysis
- Matplotlib
- Machine Learning
- Random Forest Classification
- Model Evaluation
- Feature Importance
- Kaggle Datasets
- Git and GitHub
""")